# Image Description Generation — Version Locale
Génération automatique de descriptions d'images (Flickr8k) avec PyTorch.
- Extraction de features visuelles : ResNet34 pré-entraîné
- Modèle de langue : LSTM sur les légendes Flickr8k
- Modèle combiné : image features → LSTM decoder

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from collections import Counter
from tqdm import tqdm
import re
import pickle

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")

In [ ]:
# ── Chemins locaux ──────────────────────────────────────────────────
DATA_PATH     = Path('d:/Projet_NLP_CV/Data')
IMAGES_PATH   = DATA_PATH / 'Flickr8k_Dataset' / 'Flicker8k_Dataset'
TEXT_PATH     = DATA_PATH / 'Flickr8k_text'
FEATURES_PATH = DATA_PATH / 'features'
FEATURES_PATH.mkdir(exist_ok=True)

# ── Hyperparamètres ─────────────────────────────────────────────────
BATCH_SIZE  = 48
IMAGE_SIZE  = 224
EMBED_DIM   = 256
HIDDEN_DIM  = 512
NUM_LAYERS  = 1
EPOCHS      = 10
LR          = 1e-3
MAX_LEN     = 40    # longueur max d'une légende (tokens)
MIN_FREQ    = 3     # fréquence min pour inclure un mot dans le vocab

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
print(f"Images : {IMAGES_PATH}")
print(f"Textes : {TEXT_PATH}")

## 1. Données — Flickr8k

In [ ]:
# Charger les légendes
df = pd.read_csv(TEXT_PATH / 'Flickr8k.token.txt',
                 names=['id', 'caption'], delimiter='\t')
df['number'] = df['id'].apply(lambda x: int(x.split('#')[1]))
df['id']     = df['id'].apply(lambda x: x.split('#')[0])
print(f"Total légendes : {len(df)} pour {df['id'].nunique()} images")
df.head(10)

In [ ]:
# Charger les splits train/dev/test
train_images = set(np.loadtxt(TEXT_PATH / 'Flickr_8k.trainImages.txt', dtype=str))
dev_images   = set(np.loadtxt(TEXT_PATH / 'Flickr_8k.devImages.txt',   dtype=str))
test_images  = set(np.loadtxt(TEXT_PATH / 'Flickr_8k.testImages.txt',  dtype=str))
print(f"Train : {len(train_images)} | Dev : {len(dev_images)} | Test : {len(test_images)}")

In [ ]:
# Aperçu de quelques images et leurs légendes
samples = list(test_images)[:4]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, name in zip(axes, samples):
    img = Image.open(IMAGES_PATH / name)
    cap = df[df['id'] == name]['caption'].iloc[0]
    ax.imshow(img)
    ax.set_title('\n'.join(cap[i:i+40] for i in range(0, len(cap), 40)), fontsize=8)
    ax.axis('off')
plt.suptitle('Échantillons Flickr8k', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Modèle Image — Extraction de features (ResNet34)

In [ ]:
# ResNet34 sans la couche FC finale → vecteur de 512 features
resnet = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
feature_extractor = nn.Sequential(*list(resnet.children())[:-1])  # retire FC
feature_extractor = feature_extractor.to(DEVICE)
feature_extractor.eval()
print("Feature extractor (ResNet34 sans FC) prêt.")
print(f"Sortie : (batch_size, 512, 1, 1) → squeeze → (batch_size, 512)")

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

class ImageDataset(Dataset):
    def __init__(self, image_dir, transform):
        self.image_dir  = Path(image_dir)
        self.image_files = sorted(self.image_dir.glob('*.jpg'))
        self.transform   = transform

    def __len__(self):  return len(self.image_files)

    def __getitem__(self, idx):
        path = self.image_files[idx]
        img  = Image.open(path).convert('RGB')
        return self.transform(img), path.name

img_dataset = ImageDataset(IMAGES_PATH, transform)
img_loader  = DataLoader(img_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=0)
print(f"{len(img_dataset)} images trouvées")

In [ ]:
# Extraire et sauvegarder les features (seulement si pas encore fait)
already_done = len(list(FEATURES_PATH.glob('*.pkl')))
if already_done >= len(img_dataset):
    print(f"Features déjà extraites pour {already_done} images. Saut.")
else:
    print(f"Extraction de features pour {len(img_dataset)} images...")
    with torch.no_grad():
        for images, names in tqdm(img_loader):
            images   = images.to(DEVICE)
            features = feature_extractor(images)           # (bs, 512, 1, 1)
            features = features.squeeze(-1).squeeze(-1)    # (bs, 512)
            for feat, name in zip(features.cpu(), names):
                stem = Path(name).stem
                torch.save(feat, FEATURES_PATH / f'{stem}.pkl')
    print(f"Sauvegardé : {len(list(FEATURES_PATH.glob('*.pkl')))} fichiers .pkl")

In [ ]:
# Vérification : charger et afficher la forme d'une feature
sample_name = list(test_images)[0]
sample_feat = torch.load(FEATURES_PATH / f'{Path(sample_name).stem}.pkl',
                         weights_only=True)
print(f"Feature shape pour '{sample_name}': {sample_feat.shape}")

## 3. Modèle de Langue — Vocabulaire et Dataset

In [ ]:
# Tokens spéciaux
PAD = '<pad>'
UNK = '<unk>'
SOS = '<start>'
EOS = '<end>'

def tokenize(caption):
    caption = caption.lower().strip()
    caption = re.sub(r"[^\w\s]", '', caption)
    return caption.split()

# Construire le vocabulaire
all_tokens  = [tok for cap in df['caption'] for tok in tokenize(cap)]
word_counts = Counter(all_tokens)
vocab_words = [PAD, UNK, SOS, EOS] + \
              [w for w, c in word_counts.most_common() if c >= MIN_FREQ]
word2idx = {w: i for i, w in enumerate(vocab_words)}
idx2word = {i: w for w, i in word2idx.items()}
VOCAB_SIZE = len(vocab_words)
print(f"Taille du vocabulaire : {VOCAB_SIZE} mots (fréquence >= {MIN_FREQ})")
print(f"Tokens spéciaux : PAD={word2idx[PAD]}, UNK={word2idx[UNK]}, "
      f"SOS={word2idx[SOS]}, EOS={word2idx[EOS]}")

In [ ]:
class CaptionDataset(Dataset):
    """Paires (features image, légende tokenisée) pour l'entraînement."""

    def __init__(self, image_names, df, features_path, word2idx, max_len=MAX_LEN):
        self.features_path = Path(features_path)
        self.word2idx      = word2idx
        self.max_len       = max_len
        self.pairs         = []
        for img_name in image_names:
            for cap in df[df['id'] == img_name]['caption']:
                tokens  = tokenize(cap)
                indices = ([word2idx[SOS]] +
                           [word2idx.get(t, word2idx[UNK]) for t in tokens] +
                           [word2idx[EOS]])
                self.pairs.append((img_name, indices))

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        img_name, indices = self.pairs[idx]
        stem     = Path(img_name).stem
        features = torch.load(self.features_path / f'{stem}.pkl', weights_only=True)
        indices  = indices[:self.max_len]
        cap_len  = len(indices)
        padded   = indices + [self.word2idx[PAD]] * (self.max_len - len(indices))
        return features, torch.tensor(padded, dtype=torch.long), cap_len


def collate_fn(batch):
    features, captions, lengths = zip(*batch)
    return torch.stack(features), torch.stack(captions), torch.tensor(lengths)


train_ds = CaptionDataset(train_images, df, FEATURES_PATH, word2idx)
dev_ds   = CaptionDataset(dev_images,   df, FEATURES_PATH, word2idx)
test_ds  = CaptionDataset(test_images,  df, FEATURES_PATH, word2idx)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  collate_fn=collate_fn)
dev_loader   = DataLoader(dev_ds,   batch_size=64, shuffle=False, collate_fn=collate_fn)

print(f"Train : {len(train_ds)} paires | Dev : {len(dev_ds)} | Test : {len(test_ds)}")

## 4. Modèle Combiné — Image + LSTM Decoder

In [ ]:
class ImageCaptioningModel(nn.Module):
    """
    Image → LSTM decoder.
    Les features ResNet34 (512-dim) sont projetées comme état caché initial du LSTM.
    Le décodeur génère les mots un à un (teacher forcing à l'entraînement).
    """

    def __init__(self, feature_dim, embed_dim, hidden_dim, vocab_size, num_layers=1, dropout=0.3):
        super().__init__()
        self.image_proj = nn.Linear(feature_dim, hidden_dim)
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=word2idx[PAD])
        self.lstm       = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True)
        self.fc         = nn.Linear(hidden_dim, vocab_size)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, features, captions):
        """
        features : (bs, 512)
        captions : (bs, seq_len)  — teacher forcing, inclut SOS mais pas EOS
        returns  : logits (bs, seq_len-1, vocab_size)
        """
        h0 = self.image_proj(features).unsqueeze(0)   # (1, bs, hidden)
        c0 = torch.zeros_like(h0)
        embeds  = self.dropout(self.embedding(captions[:, :-1]))  # retire le dernier token
        out, _  = self.lstm(embeds, (h0, c0))
        return self.fc(out)                            # (bs, seq_len-1, vocab_size)

    @torch.no_grad()
    def generate(self, features, max_len=MAX_LEN, device=DEVICE):
        """Génération gloutonne d'une légende à partir de features image."""
        self.eval()
        h = self.image_proj(features).unsqueeze(0)   # (1, 1, hidden)
        c = torch.zeros_like(h)
        token  = torch.tensor([[word2idx[SOS]]], device=device)
        result = []
        for _ in range(max_len):
            embed = self.embedding(token)             # (1, 1, embed)
            out, (h, c) = self.lstm(embed, (h, c))
            pred  = self.fc(out.squeeze(1)).argmax(-1)
            word  = idx2word[pred.item()]
            if word == EOS:
                break
            if word not in (PAD, UNK):
                result.append(word)
            token = pred.unsqueeze(0)
        return ' '.join(result)


model = ImageCaptioningModel(
    feature_dim=512,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    vocab_size=VOCAB_SIZE,
    num_layers=NUM_LAYERS,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Paramètres entraînables : {total_params:,}")
print(model)

## 5. Entraînement

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
criterion = nn.CrossEntropyLoss(ignore_index=word2idx[PAD])


def run_epoch(model, loader, optimizer, criterion, device, train=True):
    model.train() if train else model.eval()
    total_loss = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for features, captions, _ in tqdm(loader, leave=False):
            features = features.to(device)
            captions = captions.to(device)
            logits   = model(features, captions)          # (bs, seq-1, V)
            targets  = captions[:, 1:]                    # retire SOS
            loss     = criterion(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
            total_loss += loss.item()
    return total_loss / len(loader)


history = {'train': [], 'dev': []}
best_dev_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer, criterion, DEVICE, train=True)
    dev_loss   = run_epoch(model, dev_loader,   optimizer, criterion, DEVICE, train=False)
    scheduler.step(dev_loss)
    history['train'].append(train_loss)
    history['dev'].append(dev_loss)

    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        torch.save(model.state_dict(), DATA_PATH / 'caption_model_best.pth')
        marker = '  ← meilleur'
    else:
        marker = ''

    print(f"Epoch {epoch:2d}/{EPOCHS} | Train loss: {train_loss:.4f} | Dev loss: {dev_loss:.4f}{marker}")

print(f"\nMeilleur Dev loss : {best_dev_loss:.4f}")

In [ ]:
# Courbe d'apprentissage
plt.figure(figsize=(8, 4))
plt.plot(history['train'], label='Train loss')
plt.plot(history['dev'],   label='Dev loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.title('Courbe d\'apprentissage')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Génération de Légendes

In [ ]:
# Charger le meilleur modèle sauvegardé
model.load_state_dict(torch.load(DATA_PATH / 'caption_model_best.pth',
                                  weights_only=True, map_location=DEVICE))
model.eval()
print("Meilleur modèle chargé.")

In [ ]:
# Générer des légendes pour 6 images de test
samples = list(test_images)[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, img_name in zip(axes.flatten(), samples):
    # Afficher l'image
    img = Image.open(IMAGES_PATH / img_name)
    ax.imshow(img)

    # Générer une légende
    feat = torch.load(FEATURES_PATH / f'{Path(img_name).stem}.pkl', weights_only=True)
    feat = feat.unsqueeze(0).to(DEVICE)
    generated = model.generate(feat, device=DEVICE)

    # Légende de référence (première)
    reference = df[df['id'] == img_name]['caption'].iloc[0]

    ax.set_title(
        f"Généré : {generated}\n\nRef : {reference[:80]}",
        fontsize=8, loc='left'
    )
    ax.axis('off')

plt.suptitle('Génération automatique de légendes (test set)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Tester avec une image quelconque
def describe_image(image_path):
    """Génère une légende pour n'importe quelle image locale."""
    img  = Image.open(image_path).convert('RGB')
    x    = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        feat = feature_extractor(x).squeeze(-1).squeeze(-1)
    caption = model.generate(feat, device=DEVICE)

    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.title(f"Légende : {caption}", fontsize=10)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    return caption


# Exemple
example_img = IMAGES_PATH / list(test_images)[0]
caption = describe_image(example_img)
print(f"Légende générée : {caption}")